In [4]:
import os
import re
import pandas as pd


# ==========================================
# 1. CONFIGURATION
# ==========================================

RESUME_FOLDER = "resumes"
JOB_DESCRIPTION_FILE = "job_description.txt"

# Skills that our system can detect
SKILLS = [
    "Python",
    "Machine Learning",
    "Data Science",
    "Pandas",
    "NumPy",
    "SQL",
    "Scikit-learn",
    "Git",
    "Statistics",
    "TensorFlow",
    "Java",
    "JavaScript",
    "React",
    "HTML",
    "CSS"
]


# ==========================================
# 2. READ JOB DESCRIPTION
# ==========================================

with open(
    JOB_DESCRIPTION_FILE,
    "r",
    encoding="utf-8"
) as file:

    job_description = file.read()


print("\n========== JOB DESCRIPTION ==========")

print(job_description)


# ==========================================
# 3. FIND REQUIRED SKILLS
# ==========================================

job_skills = []

for skill in SKILLS:

    if skill.lower() in job_description.lower():

        job_skills.append(skill)


print("\n========== REQUIRED SKILLS ==========")

print(job_skills)


# ==========================================
# 4. EXTRACT CANDIDATE INFORMATION
# ==========================================

def extract_details(text):

    # ---------- NAME ----------
    name_match = re.search(
        r"Name:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if name_match:
        name = name_match.group(1).strip()
    else:
        name = "Unknown"


    # ---------- SKILLS ----------
    skills_match = re.search(
        r"Skills:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if skills_match:
        skills_text = skills_match.group(1)

        candidate_skills = []

        for skill in SKILLS:

            if skill.lower() in skills_text.lower():
                candidate_skills.append(skill)

    else:
        candidate_skills = []


    # ---------- EXPERIENCE ----------
    experience_match = re.search(
        r"Experience:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if experience_match:
        experience = experience_match.group(1).strip()
    else:
        experience = "Not mentioned"


    # ---------- EDUCATION ----------
    education_match = re.search(
        r"Education:\s*(.*)",
        text,
        re.IGNORECASE
    )

    if education_match:
        education = education_match.group(1).strip()
    else:
        education = "Not mentioned"


    return (
        name,
        candidate_skills,
        experience,
        education
    )


# ==========================================
# 5. PROCESS ALL RESUMES
# ==========================================

results = []


for filename in os.listdir(RESUME_FOLDER):

    if filename.endswith(".txt"):

        file_path = os.path.join(
            RESUME_FOLDER,
            filename
        )

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:

            resume_text = file.read()


        # Extract information
        (
            name,
            candidate_skills,
            experience,
            education
        ) = extract_details(resume_text)


        # ==================================
        # 6. MATCH SKILLS
        # ==================================

        matched_skills = []

        for skill in job_skills:

            if skill in candidate_skills:
                matched_skills.append(skill)


        # Missing skills
        missing_skills = [
            skill
            for skill in job_skills
            if skill not in candidate_skills
        ]


        # ==================================
        # 7. CALCULATE MATCH SCORE
        # ==================================

        if len(job_skills) > 0:

            skill_score = (
                len(matched_skills)
                / len(job_skills)
            ) * 100

        else:

            skill_score = 0


        # ==================================
        # 8. EXPERIENCE SCORE
        # ==================================

        experience_score = 0

        if "internship" in experience.lower():
            experience_score += 10

        if "year" in experience.lower():
            experience_score += 10


        # Limit experience score
        experience_score = min(
            experience_score,
            20
        )


        # ==================================
        # 9. FINAL SCORE
        # ==================================

        final_score = (
            skill_score * 0.8
            +
            experience_score
        )

        final_score = min(
            final_score,
            100
        )


        # ==================================
        # 10. SHORTLIST
        # ==================================

        if final_score >= 60:
            status = "Shortlisted"

        else:
            status = "Not Shortlisted"


        # ==================================
        # 11. STORE RESULT
        # ==================================

        results.append({

            "Candidate_Name": name,

            "Skills": ", ".join(
                candidate_skills
            ),

            "Matched_Skills": ", ".join(
                matched_skills
            ),

            "Missing_Skills": ", ".join(
                missing_skills
            ),

            "Experience": experience,

            "Education": education,

            "Match_Score": round(
                final_score,
                2
            ),

            "Status": status,

            "Resume_File": filename
        })


# ==========================================
# 12. CREATE DATAFRAME
# ==========================================

df = pd.DataFrame(results)


# ==========================================
# 13. RANK CANDIDATES
# ==========================================

df = df.sort_values(
    by="Match_Score",
    ascending=False
)

df["Rank"] = range(
    1,
    len(df) + 1
)


# ==========================================
# 14. DISPLAY RESULTS
# ==========================================

print("\n\n========== RESUME SCREENING RESULTS ==========")

print(
    df[
        [
            "Rank",
            "Candidate_Name",
            "Match_Score",
            "Status"
        ]
    ].to_string(index=False)
)


# ==========================================
# 15. DISPLAY MISSING SKILLS
# ==========================================

print("\n========== MISSING SKILLS ==========")

for _, row in df.iterrows():

    print(
        f"\n{row['Candidate_Name']}"
    )

    print(
        "Missing:",
        row["Missing_Skills"]
        if row["Missing_Skills"]
        else "None"
    )


# ==========================================
# 16. EXPORT SHORTLISTED CANDIDATES
# ==========================================

shortlisted = df[
    df["Status"] == "Shortlisted"
]


shortlisted.to_csv(
    "shortlisted_candidates.csv",
    index=False
)


print(
    "\n✅ Shortlisted candidates exported!"
)

print(
    "File: shortlisted_candidates.csv"
)


# ==========================================
# 17. FINAL SUMMARY
# ==========================================

print("\n========== SUMMARY ==========")

print(
    "Total Resumes:",
    len(df)
)

print(
    "Shortlisted:",
    len(shortlisted)
)

print(
    "Not Shortlisted:",
    len(df) - len(shortlisted)
)

if len(df) > 0:

    print(
        "Highest Match:",
        df.iloc[0]["Candidate_Name"]
    )

    print(
        "Highest Score:",
        df.iloc[0]["Match_Score"]
    )


print(
    "\n🎉 RESUME SCREENING COMPLETED!"
)


========== JOB DESCRIPTION ==========
Job Title: AI/ML Intern

We are looking for an AI/ML Intern with knowledge of Python, Machine Learning,
Data Science, Pandas, NumPy, SQL, and Scikit-learn.

Required Skills:
Python, Machine Learning, Data Science, Pandas, NumPy, SQL, Scikit-learn,
Git, Statistics

Education:
B.Tech / B.E. in Computer Science, Artificial Intelligence, Data Science
or related field.

Experience:
Projects or internship experience in AI/ML or Data Science is preferred.

========== REQUIRED SKILLS ==========
['Python', 'Machine Learning', 'Data Science', 'Pandas', 'NumPy', 'SQL', 'Scikit-learn', 'Git', 'Statistics']


========== RESUME SCREENING RESULTS ==========
 Rank Candidate_Name  Match_Score          Status
    1  Ananya Sharma        91.11     Shortlisted
    2    Priya Gupta        72.22     Shortlisted
    3    Arjun Singh        63.33     Shortlisted
    4    Rahul Verma        37.78 Not Shortlisted

========== MISSING SKILLS ==========

Ananya Sharma
Missing